# **Transformers**

In Transformers you send sequential data as input and get sequential data as output. For eg, machine translation, text summarization, qs and ans system etc. Mostly transformers are used in GenAI, computer vision, reinforcement learning, NLP seq2seq.

In Transfomers instead of using LSTMs or GRUs, we use a technique called `Self Attention`.

Best examples of transformers are BERT, and GPT. These are the pre-tranined transfomers.

Transformers they not only works with text data, but also with **speech**, and **image** dataset also.

**Pros**
- scalability
- transfer learning
- multimodel input output
- flexible architecture
- can easyly be intergrated with other ai tools like GAN, reinforcement learning, image captioning, etc.

**Cons**
- highly computational, GPU needed
- huge amount of data
- huge energy consumption
- lack of interpretability
- prone to biasness
- ethical concerns, coz it it is trained on data available on internet and internet data is full of biasness.


Some examples of best transformers applications : chatGPT, DALLE-2, AlphaFold, OpenAICodex, etc.

---

# **The Transformer Architecture**
### *Based on "Attention Is All You Need" (Vaswani et al., 2017)*

Before 2017, state-of-the-art Natural Language Processing (NLP) models relied on Recurrent Neural Networks (RNNs, LSTMs, GRUs). While effective, sequential models had two major flaws:
1. **Sequential Bottleneck:** Words must be processed one by one, making parallel computation on GPUs impossible during training.
2. **Vanishing / Exploding Gradients:** Long-range dependencies were hard to retain over long sentences.

The **Transformer** eliminated recurrence entirely, relying exclusively on an **Attention Mechanism** to draw global dependencies between inputs and outputs. This allowed **100% parallelized training** and revolutionized modern AI (powering GPT-4, BERT, LLaMA, and Claude).


## High-Level Architecture Overview

The Transformer follows an **Encoder-Decoder** structure:
* **Encoder:** Reads the entire input sequence (e.g., English text) and builds a rich, contextual representation for every word simultaneously.
* **Decoder:** Generates the target sequence (e.g., French translation) auto-regressively, one word at a time, using both its own previous outputs and the encoder representations.

```text
       [ Input Text ]                        [ Target Text ]
              │                                     │
      Input Embedding                       Target Embedding
              │                                     │
      Positional Encoding                   Positional Encoding
              │                                     │
              ▼                                     ▼
     ┌─────────────────┐                   ┌─────────────────┐
     │  ENCODER STACK  │                   │  DECODER STACK  │
     │   (N = 6 Layers)│                   │   (N = 6 Layers)│
     └────────┬────────┘                   └────────┬────────┘
              │                                     │
              └───────────────┐ ┌───────────────────┘
                              ▼ ▼
                       [ Linear + Softmax ]
                              │
                      Output Probabilities

---

## **1. Input Embeddings & Positional Encoding**

### **A. Input & Target Embeddings**
First, tokens (integer IDs) are mapped to dense continuous vectors of dimension $d_{\text{model}} = 512$.

### **B. Positional Encoding**
Because Transformers process all words in parallel without recurrence (unlike LSTMs), **they have no inherent sense of word order or position**. 

To inject sequence order, a **Positional Encoding vector** of the same dimension ($d_{\text{model}} = 512$) is added directly to the input embedding:

$$x_{\text{final}} = \text{Embedding}(x) + \text{PositionalEncoding}(pos)$$

The authors used sine and cosine functions of different frequencies:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i / d_{\text{model}}}}\right)$$

$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i / d_{\text{model}}}}\right)$$

### **Why Sine and Cosine?**
* Allows the model to easily learn **relative positions**, because for any fixed offset $k$, $PE_{pos+k}$ can be represented as a linear function of $PE_{pos}$.
* Can generalize to sequence lengths **longer** than those seen during training.

## **2. Scaled Dot-Product Attention (The Core Engine)**

Attention computes how much focus a specific word should pay to every other word in the sentence.

For every input vector, we project it into three vectors using learned linear weights:
* **Query ($Q$):** What the current word is looking for. (Think of it as a search query).
* **Key ($K$):** What the word offers to others. (Think of it as a video title in a database).
* **Value ($V$):** The actual content of the word. (Think of it as the actual video content).


### **The Math Equation**

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

```text
  Q   K^T
  │    │
  └──┬─┘
     ▼
[ MatMul ] ──► [ Scale by 1/√d_k ] ──► [ Mask (Opt.) ] ──► [ Softmax ] ──► [ MatMul with V ] ──► Output

## **3. Multi-Head Attention (MHA)**

Instead of computing attention just once with a single 512-dimensional vector, the Transformer splits $Q$, $K$, and $V$ into **$h = 8$ parallel attention heads**, each of dimension $d_k = 64$ ($512 / 8 = 64$).


### **Why Multi-Head Attention?**
A single head might only focus on the strongest grammatical link. Multiple heads allow the model to pay attention to different aspects of a word simultaneously:
* **Head 1:** Focuses on grammatical relationships (Subject $\rightarrow$ Verb).
* **Head 2:** Focuses on pronoun references (*"it"* $\rightarrow$ *"animal"*).
* **Head 3:** Focuses on adjective-noun pairings (*"big"* $\rightarrow$ *"dog"*).


### **The Math Equation**

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \text{head}_2, \dots, \text{head}_h) W^O$$

$$\text{where } \text{head}_i = \text{Attention}(Q W_i^Q, K W_i^K, V W_i^V)$$

```text
                   [ Input Vectors ]
                           │
       ┌───────────────────┼───────────────────┐
       ▼                   ▼                   ▼
    [ Head 1 ]          [ Head 2 ]  ...     [ Head 8 ]
   (Attention)         (Attention)         (Attention)
       │                   │                   │
       └───────────────────┼───────────────────┘
                           ▼
                 [ Concatenate Heads ]
                           │
                 [ Linear Layer (W^O) ]
                           │
                        Output


## **4. The Encoder Layer**

The Encoder consists of a stack of **$N = 6$ identical layers**. Each layer contains two main sub-layers:

1. **Sub-Layer 1:** Multi-Head Self-Attention
2. **Sub-Layer 2:** Position-wise Feed-Forward Neural Network (FFN)

Each sub-layer is wrapped with a **Residual Connection (Add)** followed by **Layer Normalization (Norm)**:

$$\text{Output} = \text{LayerNorm}\left(x + \text{SubLayer}(x)\right)$$



### **Position-wise Feed-Forward Network (FFN)**
Each position is passed independently through a 2-layer fully connected network with a ReLU activation:

$$\text{FFN}(x) = \max(0, x W_1 + b_1) W_2 + b_2$$

* Inner layer dimension: $d_{ff} = 2048$
* Output layer dimension: $d_{\text{model}} = 512$

```text
                 Input x
                    │
            ┌───────┴───────┐
            │               ▼
            │     [ Multi-Head Self-Attention ]
            │               │
            └──────►[ Add & Norm ]
                            │
                    ┌───────┴───────┐
                    │               ▼
                    │    [ Feed-Forward Network ]
                    │               │
                    └──────►[ Add & Norm ]
                            │
                     Output to next layer


## **5. The Decoder Layer**

The Decoder also consists of a stack of **$N = 6$ identical layers**. However, each layer contains **three** sub-layers:

1. **Masked Multi-Head Self-Attention:** Prevents positions from attending to subsequent (future) tokens during training. 
   * *Why Masking?* When predicting word $i$, the model must not "cheat" by looking at word $i+1$. Future positions are masked by setting their attention scores to $-\infty$ before softmax.
2. **Multi-Head Encoder-Decoder Attention (Cross-Attention):**
   * **Queries ($Q$):** Come from the Decoder's previous sub-layer.
   * **Keys ($K$) & Values ($V$):** Come from the final output of the Encoder stack.
   * This step allows every position in the Decoder to attend to all positions in the input source sentence!
3. **Position-wise Feed-Forward Network**



### **Final Output Generation**
The final output of the Decoder stack is passed through a **Linear Layer** (projecting back to vocabulary size $V$) followed by a **Softmax Layer** to output class probabilities for the next token.

## **6. Transformer Hyperparameter Summary**

Here are the exact hyperparameter configurations from the original 2017 paper (*Base Model vs. Big Model*):

| Parameter | Symbol | Base Model | Big Model |
| :--- | :---: | :---: | :---: |
| **Number of Layers** | $N$ | 6 | 6 |
| **Model Dimension** | $d_{\text{model}}$ | 512 | 1024 |
| **Attention Heads** | $h$ | 8 | 16 |
| **Head Dimension** | $d_k, d_v$ | 64 | 64 |
| **FFN Inner Dimension** | $d_{ff}$ | 2048 | 4096 |
| **Dropout** | $P_{drop}$ | 0.1 | 0.3 |
| **Warmup Steps** | - | 4000 | 4000 |



## **7. Why Transformers Conquered AI**

1. **Parallelization:** Unlike RNNs, the entire sentence is processed at once during training, maximizing GPU efficiency.
2. **Direct Long-Range Connections:** Any word can attend to any other word in a single step $O(1)$ operations, compared to $O(N)$ sequential steps in RNNs.
3. **Foundation of Modern LLMs:** 
   * **Encoder-Only:** BERT, RoBERTa (ideal for classification and embeddings).
   * **Decoder-Only:** GPT-3, GPT-4, LLaMA, Mistral (ideal for text generation).
   * **Encoder-Decoder:** T5, BART (ideal for translation and summarization).